# `test_formula.ipynb`

This notebook is a lightweight personal processing test adapted from the archived `MathNet/test_formula.ipynb`.
It targets the current in-repo `improve_tokens_unimumer.py` implementation and the committed sample data under `preprocess/examples/mathnet_hmer/`.


In [ ]:
from pathlib import Path
import sys
import json
import shutil

repo_root = Path.cwd().resolve()
while repo_root.name != "Uni-MuMER" and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if repo_root.name != "Uni-MuMER":
    raise RuntimeError("Please open this notebook from inside the Uni-MuMER repository.")

mathnet_root = repo_root / "preprocess" / "MathNet4clean"
if str(mathnet_root) not in sys.path:
    sys.path.insert(0, str(mathnet_root))

from preprocessing.improve_tokens_unimumer import improve_tokens_unimumer

sample_prompt_json = repo_root / "preprocess" / "examples" / "mathnet_hmer" / "prompt_only_output" / "prompt_only_output.json"
test_json_path = repo_root / "preprocess" / "examples" / "mathnet_hmer" / "test_formula_working.json"


In [ ]:
def format_bracket(latex_formula, image_name="1"):
    formula_tokens = latex_formula.split()
    temp_formulae = {image_name: formula_tokens}

    it = improve_tokens_unimumer(
        files=[],
        correction_files={},
        use_only_c=True,
        remove=True,
    )
    it.formulae = temp_formulae
    it.add_brackets_recursive()
    processed_formula = it.formulae[image_name]
    return " ".join(processed_formula)


def debug_improve_tokens(latex_formula, image_name="1"):
    formula_tokens = latex_formula.split()
    temp_formulae = {image_name: formula_tokens}

    it = improve_tokens_unimumer(
        files=[],
        correction_files={},
        use_only_c=True,
        remove=True,
    )
    it.formulae = temp_formulae
    print("before split_up_tokens:")
    print(" ".join(it.formulae[image_name]))
    it.split_up_tokens()
    it.remove_tokens()
    it.split_up_non_latex_commands()
    it.replace_tokens()
    it.remove_empty_sqrt_brackets()
    print("before remove_empty_brackets:")
    print(" ".join(it.formulae[image_name]))
    it.remove_empty_brackets()
    print("after remove_empty_brackets:")
    print(" ".join(it.formulae[image_name]))
    it.add_brackets_recursive()
    print("after add_brackets_recursive:")
    print(" ".join(it.formulae[image_name]))
    it.remove_empty_brackets()
    if it.remove:
        it.remove_mathtype()
        it.remove_formula_if_token()
        it.first_token_check()
    it.remove_style_elements()
    print("before simplify_recursively_uncessary_brackets:")
    print(" ".join(it.formulae[image_name]))
    it.simplify_recursively_uncessary_brackets(10)
    print("after simplify_recursively_uncessary_brackets:")
    print(" ".join(it.formulae[image_name]))
    it.order_sub_sup()
    return " ".join(it.formulae[image_name])


def proc_json_add_brackets(json_path):
    with open(json_path, "r") as f:
        json_data = json.load(f)
    backup_path = str(json_path) + ".bak"
    if not Path(backup_path).exists():
        with open(backup_path, "w") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
    else:
        with open(backup_path, "r") as f:
            json_data = json.load(f)

    for item in json_data:
        gt = item["messages"][-1]["value"]
        try:
            item["messages"][-1]["value"] = format_bracket(gt)
        except Exception:
            item["messages"][-1]["value"] = "<ERROR>"
            print(json.dumps(item, indent=4, ensure_ascii=False))

    with open(json_path, "w") as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)


In [ ]:
latex_formula = r"a ^ 2"
print(latex_formula)
print(format_bracket(latex_formula))
print(debug_improve_tokens(latex_formula))


In [ ]:
latex_formula = r"\\Theta [ \\begin {array} { c } \\alpha \\\\ \\beta \\end{array} ] ( z | \\tau ) = \\sum _ { n = - \\infty } ^ { + \\infty } e ^ { i \\pi \\tau ( n + \\alpha ) ^ { 2 } + 2 i \\pi ( n + \\alpha ) ( z + \\beta ) }"
print(latex_formula)
print(debug_improve_tokens(latex_formula))


In [ ]:
latex_formula = r"\\rho \\frac { \\operatorname { d } v } { \\operatorname { d } t } = - \\frac { \\operatorname { d } p } { \\operatorname { d } x }"
print(latex_formula)
print(debug_improve_tokens(latex_formula))


In [ ]:
if test_json_path.exists():
    test_json_path.unlink()
shutil.copy(sample_prompt_json, test_json_path)
proc_json_add_brackets(test_json_path)
print(test_json_path)
print(test_json_path.with_suffix(test_json_path.suffix + '.bak'))


In [ ]:
with open(test_json_path, "r") as f:
    preview = json.load(f)
preview[:2]
